# Local Data Workflow — dlt + DuckDB + Jupyter

This notebook **orchestrates** the pipeline; it does not define business logic.
Every function called here lives in the versioned, testable `pipeline.py` module.
That separation is deliberate: notebooks are great for exploration and terrible
for hiding production logic (no diffs, no unit tests, hidden execution order).

**Flow:** extract → quality gate → load (dlt→DuckDB) → analyze (SQL) → export.

## 0. Setup
Import the pipeline module and make output wide enough to read.

In [1]:
import sys
from pathlib import Path

# Make the project root importable regardless of where Jupyter is launched.
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd

from demos import pipeline

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
print("pipeline module loaded from:", pipeline.__file__)

pipeline module loaded from: /home/claude/data-pipeline-platform/demos/pipeline.py


## 1. Extract
Read the source CSV with **explicit** types (never trust inference for money or
dates) and derive `revenue = quantity * unit_price` as a first-class measure.

In [2]:
df = pipeline.load_raw_sales()
print(df.dtypes, "\n")
df

order_id                int64
order_date     datetime64[us]
customer_id               str
product                   str
category                  str
region                    str
quantity                int64
unit_price            float64
revenue               float64
dtype: object 



,order_id,order_date,customer_id,product,category,region,quantity,unit_price,revenue
0,1001,2024-01-05,C017,Wireless Mouse,Accessories,North,3,24.99,74.97
1,1002,2024-01-06,C042,Mechanical Keyboard,Accessories,West,1,89.50,89.50
2,1003,2024-01-08,C019,27-inch Monitor,Displays,North,2,219.00,438.00
3,1004,2024-01-11,C007,USB-C Hub,Accessories,South,5,39.95,199.75
4,1005,2024-01-15,C042,Laptop Stand,Accessories,West,2,45.00,90.00
5,1006,2024-01-18,C033,34-inch Ultrawide,Displays,East,1,499.99,499.99
6,1007,2024-01-21,C051,Webcam 1080p,Peripherals,South,4,62.75,251.00
7,1008,2024-01-24,C007,Noise-Cancelling Headset,Peripherals,South,1,129.00,129.00
8,1009,2024-01-28,C060,Docking Station,Accessories,East,2,159.99,319.98
9,1010,2024-01-30,C019,4K Monitor,Displays,North,1,389.00,389.00


## 2. Quality gate
Validate business rules **before** loading. Loading dirty data "to clean later"
is how silent corruption gets into a warehouse. If any check fails, we stop.

In [3]:
report = pipeline.run_quality_checks(df)
print(report.summary())
assert report.passed, "Quality gate failed — investigate before loading."

Rows evaluated: 10
  [PASS] all_required_columns_present
  [PASS] no_nulls_in_required_fields
  [PASS] order_id_is_unique
  [PASS] quantity_positive
  [PASS] unit_price_positive
  [PASS] order_date_parses


### Demonstrating the gate catching bad data
A check is only trustworthy if you've seen it fail. Here we inject a duplicate
primary key and a negative price into a copy and confirm the gate rejects it.

In [4]:
bad = df.copy()
bad.loc[0, "order_id"] = bad.loc[1, "order_id"]   # duplicate PK
bad.loc[2, "unit_price"] = -5.0                    # invalid price

bad_report = pipeline.run_quality_checks(bad)
print(bad_report.summary())
print("\nGate passed?", bad_report.passed)   # -> False, as intended

Rows evaluated: 10
  [PASS] all_required_columns_present
  [PASS] no_nulls_in_required_fields
  [FAIL] order_id_is_unique
  [PASS] quantity_positive
  [FAIL] unit_price_positive
  [PASS] order_date_parses
Errors:
    - 1 duplicate order_id value(s)
    - Non-positive unit_price detected

Gate passed? False


## 3. Load into DuckDB via dlt
`dlt` infers and versions the schema, adds lineage columns, and namespaces the
table under its dataset. `write_disposition="replace"` keeps re-runs idempotent.

In [5]:
pipe, load_info = pipeline.load_to_duckdb(df)
print(load_info)

Pipeline local_sales load step completed in 0.24 seconds
1 load package(s) were loaded to destination duckdb and into dataset sales
The duckdb destination used duckdb:////home/claude/data-pipeline-platform/output/local_sales.duckdb location to store data
Load package 1785969859.1249409 is LOADED and contains no failed jobs


### Inspecting the schema dlt created (schema-awareness)
dlt lands our table at **`sales.orders`** and adds two lineage columns:
`_dlt_load_id` (which load produced the row) and `_dlt_id` (a stable row hash).
It also maintains internal `_dlt_*` tables for load history and state. This is
why raw SQL must qualify the table as `sales.orders`, not just `orders`.

In [6]:
schema_df = pipeline.query('''
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'sales' AND table_name = 'orders'
    ORDER BY ordinal_position
''')
schema_df

,column_name,data_type
0,order_id,BIGINT
1,order_date,TIMESTAMP WITH TIME ZONE
2,customer_id,VARCHAR
3,product,VARCHAR
4,category,VARCHAR
5,region,VARCHAR
6,quantity,BIGINT
7,unit_price,DOUBLE
8,revenue,DOUBLE
9,_dlt_load_id,VARCHAR


## 4. Analytics
SQL executed against the dlt-managed dataset, returned as DataFrames.

### 4.1 Summary statistics

In [7]:
pipeline.summary_statistics()

,order_count,units_sold,total_revenue,avg_order_value,unique_customers
0,10,22.0,2481.19,248.12,7


### 4.2 Revenue by category

In [8]:
pipeline.revenue_by_category()

,category,orders,revenue
0,Displays,3,1326.99
1,Accessories,5,774.20
2,Peripherals,2,380.00


### 4.3 Revenue by region

In [9]:
pipeline.revenue_by_region()

,region,orders,revenue
0,North,3,901.97
1,East,2,819.97
2,South,3,579.75
3,West,2,179.50


### 4.4 Ad-hoc query
The `pipeline.query()` helper opens a **read-only** connection, so exploratory
SQL can never mutate the warehouse.

In [10]:
pipeline.query('''
    SELECT product, region, revenue
    FROM sales.orders
    ORDER BY revenue DESC
    LIMIT 5
''')

,product,region,revenue
0,34-inch Ultrawide,East,499.99
1,27-inch Monitor,North,438.00
2,4K Monitor,North,389.00
3,Docking Station,East,319.98
4,Webcam 1080p,South,251.00


## 5. Export
Persist every result to `output/` as CSV for downstream consumers.

In [11]:
paths = pipeline.export_all()
for p in paths:
    print("wrote", p.name)

wrote summary_statistics.csv
wrote revenue_by_category.csv
wrote revenue_by_region.csv


## 6. Where this goes next
- Swap `write_disposition` to `"merge"` with a primary key for incremental loads.
- Point `dlt` at a `rest_api` or SQL source instead of a static CSV.
- Change the destination from `duckdb` to `postgres`/`bigquery` — one line.
- Wrap `test_setup.py` + `pipeline.main()` in CI to catch regressions.